In [2]:
from pyspark.sql import SparkSession

PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Permormance_Analytics") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [3]:
from pyspark.sql.functions import *

spark.sql("""
SELECT 
    COUNT(*) as total_files,
    ROUND(AVG(file_size_in_bytes) / 1024, 2) as avg_size_kb,
    SUM(record_count) as total_records
FROM my_catalog.default.user_events.files
""").show()

spark.sql("""
SELECT 
    file_path, 
    file_size_in_bytes, 
    record_count 
FROM my_catalog.default.user_events.files 
LIMIT 5
""").show(truncate=False)

+-----------+-----------+-------------+
|total_files|avg_size_kb|total_records|
+-----------+-----------+-------------+
|       9000|      42.66|     14889721|
+-----------+-----------+-------------+

+----------------------------------------------------------------------------------------------------+------------------+------------+
|file_path                                                                                           |file_size_in_bytes|record_count|
+----------------------------------------------------------------------------------------------------+------------------+------------+
|s3a://warehouse/default/user_events/data/00000-12-d7befe18-2bf3-4e8e-a14a-8c4aff00b245-00001.parquet|43785             |1654        |
|s3a://warehouse/default/user_events/data/00001-13-d7befe18-2bf3-4e8e-a14a-8c4aff00b245-00001.parquet|43544             |1654        |
|s3a://warehouse/default/user_events/data/00002-14-d7befe18-2bf3-4e8e-a14a-8c4aff00b245-00001.parquet|43568             |165

In [11]:
import time
import os 
from datetime import datetime

LOG_FILE = "benchmark_results_compact2.csv"

if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        f.write("timestamp,files_count,execution_time_agg,execution_time_single\n")
    print(f"log file created: {LOG_FILE}")

def benchmark_read(table_name="my_catalog.default.user_events"):
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"run ts: {current_time}")

    
    start_time_agg = time.time()
    spark.sql(f"""
        SELECT 
            shop,
            device_type,
            event_type,
            COUNT(DISTINCT session_id) as unique_sessions, 
            COUNT(DISTINCT user_id) as unique_users,
            SUM(price) as total_revenue, 
            AVG(price) as avg_check,
            MIN(timestamp) as first_interaction,
            MAX(timestamp) as latest_interaction
        FROM {table_name}
        WHERE shop IN ('rozetka', 'foxtrot', 'citrus') AND device_type IN ('ios', 'android')
        GROUP BY 1,2,3
        ORDER BY 
            total_revenue DESC,
            unique_sessions DESC
    """).collect() # action for spark to actually calculate things
    
    end_time_agg = time.time()
    execution_time_agg = end_time_agg - start_time_agg
    files_count = spark.sql("SELECT COUNT(*) FROM my_catalog.default.user_events.files").collect()[0][0]

    start_time_single = time.time()
    spark.sql(f"""
        SELECT * FROM {table_name}
        WHERE shop = 'rozetka' 
          AND device_type = 'mobile_web' 
          AND event_type = 'purchase'
    """).collect()
    end_time_single = time.time()

    execution_time_single = end_time_single - start_time_single

    with open(LOG_FILE, "a") as f:
        f.write(f"{current_time},{files_count},{execution_time_agg:.4f},{execution_time_single:.4f}\n")
    
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (AGG): {execution_time_agg:.4f}")
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (SINGLE): {execution_time_single:.4f}")


In [14]:
benchmark_read()

run ts: 2026-05-20 20:46:23
FILES NUMBER: 1 /// TIME EXECUTING (AGG): 11.2794
FILES NUMBER: 1 /// TIME EXECUTING (SINGLE): 3.0002


In [34]:
from datetime import datetime
LOG_FILE = "benchmark_results.csv"
current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        f.write("timestamp,files_count,execution_time_agg,execution_time_single\n")
    print(f"log file created: {LOG_FILE}")

with open(LOG_FILE, "a") as f:
    f.write(f"{current_time},52,1.9935,0.8688\n")
    f.write(f"{current_time},137,3.9046,1.6432\n")
    f.write(f"{current_time},303,7.3731,3.5759\n")
    f.write(f"{current_time},661,18.1840,5.2181\n")
    print("123123")

log file created: benchmark_results.csv
123123


In [13]:
import time

start_time = time.time()

result = spark.sql("""
    CALL my_catalog.system.rewrite_data_files(
        table => 'default.user_events'
    )
""")

result.show(truncate=False)

execution_time = time.time() - start_time
print(execution_time)

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|9000                      |1                     |393172793            |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

97.54932403564453


In [3]:

INTERVAL_MINUTES = 30

try:
    while True:
        benchmark_read()
        print(f"waiting set interval")
        time.sleep(INTERVAL_MINUTES * 60)
except KeyboardInterrupt:
    print("interrupted")

NameError: name 'benchmark_read' is not defined

In [17]:
spark.sql("""
SELECT 
    COUNT(*) as total
FROM my_catalog.default.user_events
""").show()

+--------+
|   total|
+--------+
|14889721|
+--------+



In [9]:
spark.sql("SELECT * FROM my_catalog.default.user_events.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-20 20:25:04.968|8740935399334946007|NULL               |true               |
|2026-05-20 20:34:11.561|8877305428001306899|8740935399334946007|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [10]:
spark.sql("""
    CALL my_catalog.system.rollback_to_snapshot(
        table => 'default.user_events',
        snapshot_id => 8740935399334946007 
    )
""")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [15]:
spark.sql("DROP TABLE IF EXISTS my_catalog.default.user_events_sorted")

spark.sql("""
CREATE TABLE my_catalog.default.user_events_sorted (
    user_id INT,
    shop STRING,
    event_type STRING,
    device_type STRING,
    price DOUBLE,
    session_id STRING,
    timestamp TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
)
""")

spark.sql("ALTER TABLE my_catalog.default.user_events_sorted WRITE ORDERED BY shop, device_type, event_type")

print("654")

654
